# 04 · Preference tuning after SFT

Use DPO only on preferences whose chosen answer is demonstrably
better under the same harness and hidden verifier. This stage is
intentionally smaller than SFT and cannot repair a broken tool schema.

## Install and authenticate

In [ ]:
import subprocess
import sys
from pathlib import Path

# The Git pins supply current Unsloth/Qwen3.8 support. Transformers, TRL and
# Datasets deliberately use the mutually compatible versions from the adjacent
# official Unsloth Qwen3.5 27B notebook. Do not replace these with branch-head
# SHAs without resolving package metadata together first.
GIT_REVISIONS = {
    "unsloth": "c87fe20e32aca9ceb2dc5059c2987738f32446e8",
    "unsloth_zoo": "5b239e574f03ab3077c17e49aeef3cacfe7cdd4e",
}

import torch

torch_version = torch.__version__.split("+", 1)[0]
torch_minor = ".".join(torch_version.split(".")[:2])
torchao_by_torch = {"2.8": "0.16.0", "2.9": "0.16.0", "2.10": "0.16.0", "2.11": "0.18.0"}
xformers_by_torch = {"2.8": "0.0.32.post2", "2.9": "0.0.33.post1", "2.10": "0.0.34", "2.11": "0.0.34"}
if torch_minor not in torchao_by_torch:
    raise RuntimeError(
        f"No reviewed Colab dependency set for torch {torch.__version__}. "
        f"Expected one of {sorted(torchao_by_torch)}; update the compatibility matrix first."
    )

COMPATIBILITY_PINS = {
    "transformers": "5.3.0",
    "trl": "0.22.2",
    "datasets": "4.3.0",
    "peft": "0.19.0",
    "torchao": torchao_by_torch[torch_minor],
    "xformers": xformers_by_torch[torch_minor],
}
INSTALLER_REVISION = "colab-v2"
pin_key = "-".join(value.replace(".", "") for value in COMPATIBILITY_PINS.values())
git_key = "-".join(value[:8] for value in GIT_REVISIONS.values())
INSTALL_KEY = f"{INSTALLER_REVISION}-torch{torch_minor}-{git_key}-{pin_key}"
INSTALL_MARKER = Path(f"/content/.qwen38_env_{INSTALL_KEY}")
PIP_LOG = Path("/content/qwen38_pip_install.log")
FORCE_INSTALL = False

def install_phase(name: str, packages: list[str], *, no_deps: bool = False) -> None:
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        "--no-cache-dir",
        "--log",
        str(PIP_LOG),
    ]
    if no_deps:
        command.append("--no-deps")
    command.extend(packages)
    print(f"\n=== install phase: {name} ===")
    print("\n".join(f"  {package}" for package in packages))
    result = subprocess.run(command, check=False)
    if result.returncode:
        log_tail = (
            "\n".join(PIP_LOG.read_text(errors="replace").splitlines()[-120:])
            if PIP_LOG.exists()
            else "[pip did not create its log file]"
        )
        print(f"\n--- tail of {PIP_LOG} ---\n{log_tail}")
        raise RuntimeError(
            f"Package installation failed during {name!r} with exit code {result.returncode}. "
            f"The detailed log is at {PIP_LOG}."
        )

if FORCE_INSTALL or not INSTALL_MARKER.exists():
    if PIP_LOG.exists():
        PIP_LOG.unlink()
    install_phase("packaging tools", ["pip", "setuptools==80.9.0", "wheel>=0.42.0"])
    install_phase("Qwen3.8 training stack", [
        f"unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git@{GIT_REVISIONS['unsloth_zoo']}",
        f"unsloth @ git+https://github.com/unslothai/unsloth.git@{GIT_REVISIONS['unsloth']}",
        f"torch=={torch_version}",
        f"torchao=={COMPATIBILITY_PINS['torchao']}",
        f"transformers=={COMPATIBILITY_PINS['transformers']}",
        f"trl=={COMPATIBILITY_PINS['trl']}",
        f"datasets=={COMPATIBILITY_PINS['datasets']}",
        f"peft=={COMPATIBILITY_PINS['peft']}",
        "accelerate",
        "bitsandbytes",
        "trackio",
        "huggingface_hub>=0.34.0,<2.0",
        "hf_transfer",
        "sentencepiece>=0.2.0",
        "protobuf",
        "pytest",
        "jmespath",
    ])
    install_phase(
        "PyTorch-matched xFormers wheel",
        [f"xformers=={COMPATIBILITY_PINS['xformers']}"],
        no_deps=True,
    )
    INSTALL_MARKER.write_text(INSTALL_KEY)
    print("Packages installed. Restart the Colab runtime, then rerun this notebook from the top.")
else:
    print(f"Pinned environment already installed: {INSTALL_KEY}")

After the first install, restart the runtime and rerun the notebook from the top; the install marker skips the pip work.

In [ ]:
import json
import os
import platform
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import torch
from huggingface_hub import login, whoami

if "GIT_REVISIONS" not in globals():
    raise RuntimeError(
        "This runtime was restarted. Rerun the notebook from the first cell; "
        "the install marker will skip the expensive package installation."
    )
if "COMPATIBILITY_PINS" not in globals():
    raise RuntimeError("Missing compatibility pins; rerun the notebook from the first cell.")

try:
    from google.colab import userdata
except ImportError:
    userdata = None

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab G4 GPU runtime before continuing.")

gpu = torch.cuda.get_device_properties(0)
gpu_total_gib = gpu.total_memory / 1024**3
# A vendor-labelled 96 GB card can be reported as about 89.4 GiB because
# PyTorch converts the byte count with a binary divisor. Keep the floor well
# above the roughly 44.7 GiB reported for a 48 GB card without rejecting G4.
MIN_G4_TOTAL_GIB = 85.0
print(
    f"GPU: {gpu.name} ({gpu_total_gib:.1f} GiB total), "
    f"capability={torch.cuda.get_device_capability(0)}"
)
if gpu_total_gib < MIN_G4_TOTAL_GIB:
    raise RuntimeError(
        "This suite expects the nominal 96 GB Colab G4 runtime. "
        f"PyTorch reports {gpu_total_gib:.1f} GiB total; expected at least "
        f"{MIN_G4_TOTAL_GIB:.0f} GiB. A value near 45 GiB usually indicates "
        "the 48 GB GPU variant."
    )

hf_token = userdata.get("HF_TOKEN") if userdata is not None else os.getenv("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add a write-capable HF_TOKEN to Colab Secrets before continuing.")
login(token=hf_token, add_to_git_credential=False)
HF_USERNAME = whoami()["name"]

def package_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "missing"

observed_pins = {name: package_version(name) for name in COMPATIBILITY_PINS}
pin_mismatches = {
    name: {"expected": expected, "observed": observed_pins[name]}
    for name, expected in COMPATIBILITY_PINS.items()
    if observed_pins[name] != expected
}
if pin_mismatches:
    raise RuntimeError(
        "The runtime does not match the reviewed compatibility set. "
        f"Rerun the install cell with FORCE_INSTALL=True: {pin_mismatches}"
    )

RUN_ROOT = Path("/content/qwen38_runs")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
runtime_manifest = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": gpu.name,
    "gpu_total_gib": round(gpu_total_gib, 2),
    "packages": {
        name: package_version(name)
        for name in ["unsloth", "unsloth_zoo", "transformers", "trl", "peft", "datasets"]
    },
    "git_revisions": GIT_REVISIONS,
    "compatibility_pins": COMPATIBILITY_PINS,
}
(RUN_ROOT / "runtime_manifest.json").write_text(json.dumps(runtime_manifest, indent=2))
print(json.dumps(runtime_manifest, indent=2))
print(f"Authenticated as {HF_USERNAME}")

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset, load_dataset
from trl import DPOConfig, DPOTrainer

SFT_ADAPTER_ID = f"{HF_USERNAME}/qwen38-27b-code-sft-lora"
SFT_ADAPTER_REVISION = "REPLACE_WITH_ACCEPTED_COMMIT"
PREFERENCE_DATASET_ID = f"{HF_USERNAME}/qwen38-code-preferences"
PREFERENCE_DATASET_REVISION = "main"
OUTPUT_ADAPTER_ID = f"{HF_USERNAME}/qwen38-27b-code-dpo-lora"
MAX_SEQ_LENGTH = 4_096
MAX_STEPS = 2
DEMO_MODE = True
RUN_TRAINING = False
PUSH_ADAPTER = False

if RUN_TRAINING and SFT_ADAPTER_REVISION.startswith("REPLACE_"):
    raise RuntimeError("Pin the accepted SFT adapter commit before DPO.")
if DEMO_MODE and (PUSH_ADAPTER or MAX_STEPS > 2):
    raise RuntimeError("Demo preferences are limited to two local smoke steps and cannot be published.")

## Load the accepted SFT adapter

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER_ID,
    revision=None if SFT_ADAPTER_REVISION.startswith("REPLACE_") else SFT_ADAPTER_REVISION,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.bfloat16,
    load_in_4bit=False,
    token=hf_token,
)
tokenizer.padding_side = "left"
trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
total = sum(parameter.numel() for parameter in model.parameters())
if not trainable:
    raise RuntimeError("The SFT adapter loaded without trainable parameters; inspect PEFT loading before DPO.")
print({"trainable": trainable, "total": total, "fraction": trainable / total})

## Render prompt/chosen/rejected with the native Qwen3.8 template

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List files below a repository-relative directory.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a UTF-8 repository file with bounded output.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search repository text using a regular expression.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "apply_patch",
            "description": "Apply a unified diff to files inside the repository.",
            "parameters": {
                "type": "object",
                "properties": {"patch": {"type": "string"}},
                "required": ["patch"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_tests",
            "description": "Run an allow-listed repository test profile.",
            "parameters": {
                "type": "object",
                "properties": {"profile": {"type": "string", "enum": ["unit"]}},
                "required": ["profile"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "shell",
            "description": "Run a restricted allow-listed command. It is disabled in the pilot.",
            "parameters": {
                "type": "object",
                "properties": {"command": {"type": "string"}},
                "required": ["command"],
                "additionalProperties": False,
            },
        },
    },
]

def _without_arrow_nulls(value):
    """Remove null struct fields inserted by a Datasets/Arrow round trip."""
    if isinstance(value, dict):
        cleaned = {}
        for key, item in value.items():
            normalized = _without_arrow_nulls(item)
            if normalized is not None:
                cleaned[key] = normalized
        return cleaned
    if isinstance(value, list):
        return [_without_arrow_nulls(item) for item in value]
    return value

def canonical_tool_schema(tools: list[dict]) -> str:
    """Return a stable semantic fingerprint while retaining tool order."""
    return json.dumps(
        _without_arrow_nulls(tools),
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    )

TOOL_SCHEMA_JSON = canonical_tool_schema(TOOLS)

def canonical_to_qwen(messages: list[dict]) -> list[dict]:
    """Fold an initial developer message into system for the HF tokenizer.

    The adapter performs the same mapping in training and deployment. The
    official safetensor tokenizer currently accepts system/user/assistant/tool.
    """
    converted = []
    pending_system = []
    for stored_message in messages:
        message = _without_arrow_nulls(stored_message)
        role = message["role"]
        if role in {"system", "developer"} and not converted:
            pending_system.append(str(message.get("content", "")))
            continue
        if pending_system:
            converted.append({"role": "system", "content": "\n\n".join(pending_system)})
            pending_system = []
        converted.append(message)
    if pending_system:
        converted.append({"role": "system", "content": "\n\n".join(pending_system)})
    return converted

def render_chat(messages: list[dict], *, add_generation_prompt: bool) -> str:
    return tokenizer.apply_chat_template(
        canonical_to_qwen(messages),
        tools=TOOLS,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
        enable_thinking=True,
        reasoning_effort="medium",
        preserve_thinking=True,
    )

In [ ]:
demo_preferences = Dataset.from_list([
    {
        "prompt_messages": [
            {"role": "developer", "content": "Make the smallest correct change and report verification."},
            {"role": "user", "content": "The bounds check is inverted; what did you change?"},
        ],
        "chosen_message": {"role": "assistant", "content": "Corrected only the inverted comparison and verified the focused unit tests pass."},
        "rejected_message": {"role": "assistant", "content": "Rewrote the entire module and skipped tests."},
        "chosen_reward": 1.0,
        "rejected_reward": 0.0,
        "infra_status": "ok",
    },
    {
        "prompt_messages": [
            {"role": "developer", "content": "Inspect evidence before proposing a patch."},
            {"role": "user", "content": "A parser test fails only for empty input."},
        ],
        "chosen_message": {"role": "assistant", "content": "I would first inspect the failing test and empty-input branch before editing."},
        "rejected_message": {"role": "assistant", "content": "Delete the failing test."},
        "chosen_reward": 1.0,
        "rejected_reward": 0.0,
        "infra_status": "ok",
    },
])

USE_DEMO_DATA = DEMO_MODE
if USE_DEMO_DATA:
    raw = demo_preferences
    print("Using synthetic plumbing preferences; this is not a capability run.")
else:
    raw = load_dataset(
        PREFERENCE_DATASET_ID,
        split="train",
        revision=PREFERENCE_DATASET_REVISION,
        token=hf_token,
    )

def render_preference(row):
    if row.get("infra_status") != "ok":
        raise ValueError("Infrastructure failures must not become preferences.")
    if not row["chosen_reward"] > row["rejected_reward"]:
        raise ValueError("Chosen reward must be strictly greater than rejected reward.")
    prompt = canonical_to_qwen(row["prompt_messages"])
    prompt_text = tokenizer.apply_chat_template(
        prompt,
        tools=TOOLS,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
        reasoning_effort="medium",
    )

    def completion(message):
        full = tokenizer.apply_chat_template(
            prompt + [message],
            tools=TOOLS,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=True,
            reasoning_effort="medium",
            preserve_thinking=True,
        )
        if not full.startswith(prompt_text):
            raise ValueError("Template prefix drift: chosen/rejected cannot be separated safely.")
        return full[len(prompt_text):]

    return {
        "prompt": prompt_text,
        "chosen": completion(row["chosen_message"]),
        "rejected": completion(row["rejected_message"]),
    }

preferences = raw.map(render_preference, remove_columns=raw.column_names)
split = preferences.train_test_split(test_size=0.5 if len(preferences) < 20 else 0.1, seed=3407)
print({key: split["train"][0][key][:1000] for key in ["prompt", "chosen", "rejected"]})

## Configure and optionally run DPO

In [ ]:
dpo_args = DPOConfig(
    output_dir=str(RUN_ROOT / "dpo"),
    max_length=MAX_SEQ_LENGTH,
    beta=0.1,
    loss_type="sigmoid",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-7,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    max_steps=MAX_STEPS,
    bf16=True,
    optim="adamw_8bit",
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=1,
    save_strategy="steps",
    save_steps=1,
    save_total_limit=2,
    precompute_ref_log_probs=True,
    report_to="trackio",
    run_name="qwen38-code-dpo-smoke" if USE_DEMO_DATA else "qwen38-code-dpo",
    push_to_hub=PUSH_ADAPTER,
    hub_model_id=OUTPUT_ADAPTER_ID,
    hub_strategy="every_save",
    seed=3407,
)
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_args,
    processing_class=tokenizer,
    train_dataset=split["train"],
    eval_dataset=split["test"],
)

if RUN_TRAINING:
    result = trainer.train()
    trainer.save_model(str(RUN_ROOT / "dpo" / "final_adapter"))
    if PUSH_ADAPTER:
        trainer.push_to_hub(commit_message="DPO adapter from verifier-backed preferences")
    print(result.metrics)
else:
    print("DPO dry run configured. Inspect rendered pairs before setting RUN_TRAINING=True.")

## Acceptance gate

Compare the DPO adapter against the accepted SFT adapter on the
same frozen tasks. Reject it if patch correctness, native tool
validity, or reasoning-retention policy regresses—even if the
preference objective improves.